# Sentiment Stability Analysis

This notebook loads the production sentiment stability feature set and yearly market returns, merges them by year, and evaluates whether sentiment deviation is related to forward returns.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def get_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == 'notebooks' else cwd

BASE_DIR = get_project_root()
if str(BASE_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(BASE_DIR / 'src'))

STABILITY_PATH = BASE_DIR / 'features' / 'sentiment_stability.csv'
MARKET_PATH = BASE_DIR / 'features' / 'market_returns.csv'

print(f'Loading sentiment stability features from: {STABILITY_PATH}')
print(f'Loading market returns from: {MARKET_PATH}')

stability = pd.read_csv(STABILITY_PATH)
market_returns = pd.read_csv(MARKET_PATH)

stability['year'] = pd.to_numeric(stability['year'], errors='coerce').astype('Int64')
market_returns['year'] = pd.to_numeric(market_returns['year'], errors='coerce').astype('Int64')

stability = stability.sort_values(['company_id', 'year'], kind='mergesort').reset_index(drop=True)
market_returns = market_returns.sort_values('year', kind='mergesort').reset_index(drop=True)

print('stability rows:', len(stability))
print('market rows:', len(market_returns))
print('stability year range:', stability['year'].min(), '->', stability['year'].max())
print('market year range:', market_returns['year'].min(), '->', market_returns['year'].max())


In [ ]:
analysis_df = stability.merge(market_returns, on='year', how='left', validate='many_to_one')
analysis_df = analysis_df.dropna(subset=['sentiment_deviation', 'next_year_return']).copy()
analysis_df = (
    analysis_df
    .groupby('year', as_index=False)
    .agg(
        sentiment_deviation=('sentiment_deviation', 'mean'),
        next_year_return=('next_year_return', 'first')
    )
    .sort_values('year', kind='mergesort')
    .reset_index(drop=True)
)

unique_years = analysis_df['year'].nunique()
row_count = len(analysis_df)
duplicate_years = analysis_df.loc[analysis_df['year'].duplicated(), 'year'].tolist()

print('merged rows after yearly aggregation:', row_count)
print('unique years count:', unique_years)
print('confirm 1 row per year:', row_count == unique_years)

if duplicate_years:
    raise ValueError(f'Duplicate years remain after aggregation: {duplicate_years}')

analysis_df['stability_regime'] = pd.qcut(
    analysis_df['sentiment_deviation'],
    q=3,
    labels=['stable', 'neutral', 'unstable'],
    duplicates='drop',
)

analysis_df.head()


In [ ]:
correlation = analysis_df['sentiment_deviation'].corr(analysis_df['next_year_return'])

regime_summary = (
    analysis_df.assign(
        is_negative=analysis_df['next_year_return'] < 0,
        is_above_20=analysis_df['next_year_return'] > 0.20,
    )
    .groupby('stability_regime', observed=False)
    .agg(
        observations=('next_year_return', 'size'),
        mean_return=('next_year_return', 'mean'),
        median_return=('next_year_return', 'median'),
        std_return=('next_year_return', 'std'),
        negative_return_pct=('is_negative', 'mean'),
        above_20pct_return_pct=('is_above_20', 'mean'),
    )
)

regime_order = ['stable', 'neutral', 'unstable']
regime_summary = regime_summary.reindex(regime_order)
regime_summary[['negative_return_pct', 'above_20pct_return_pct']] = (
    regime_summary[['negative_return_pct', 'above_20pct_return_pct']].mul(100)
)

regime_summary['decision_insight'] = pd.Series(
    {
        'stable': 'favorable environment',
        'neutral': 'mixed environment',
        'unstable': 'risk elevated',
    }
)

print('correlation (deviation vs forward returns):', round(correlation, 4))
display(regime_summary.round(4))


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    analysis_df['sentiment_deviation'],
    analysis_df['next_year_return'].abs(),
    alpha=0.7,
)
plt.title('Sentiment Deviation vs Absolute Forward Returns')
plt.xlabel('Sentiment Deviation')
plt.ylabel('|Next-Year Return|')
plt.tight_layout()
plt.show()


In [ ]:
regime_plot_df = (
    analysis_df.assign(
        regime_level=analysis_df['stability_regime'].map(
            {'stable': 0, 'neutral': 1, 'unstable': 2}
        )
    )
    .sort_values('year', kind='mergesort')
)

fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.step(
    regime_plot_df['year'],
    regime_plot_df['regime_level'],
    where='mid',
    color='tab:blue',
    linewidth=2,
    label='Stability Regime',
)
ax1.set_xlabel('Year')
ax1.set_ylabel('Stability Regime', color='tab:blue')
ax1.set_yticks([0, 1, 2])
ax1.set_yticklabels(['stable', 'neutral', 'unstable'])
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.plot(
    regime_plot_df['year'],
    regime_plot_df['next_year_return'],
    color='tab:green',
    marker='o',
    linewidth=2,
    label='Next-Year Return',
)
ax2.axhline(0, color='tab:gray', linestyle='--', linewidth=1)
ax2.set_ylabel('Next-Year Return', color='tab:green')
ax2.tick_params(axis='y', labelcolor='tab:green')

lines = ax1.get_lines() + ax2.get_lines()
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc='best')
plt.title('Stability Regimes Over Time with Forward Returns')
plt.tight_layout()
plt.show()

regime_plot_df[['year', 'stability_regime', 'next_year_return']].tail()
